In [1]:
import os 
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.express as px
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from random import choice
import h3

from mirrorverse.utils import read_data_w_cache

In [2]:
model = '3_1_18'
run_id = 'dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed550adc8c2c38a11d896'
data = read_data_w_cache(
    f"select *, month(time) as month from chinook_depth_inference_{model} where run_id = '{run_id}' and not _train"
)
data['max_depth_bin'] = data.groupby(['_individual', '_decision'])['n_depth_bin'].transform('max')
data['radians'] = np.arctan2(data['sin_sun'], data['cos_sun'])
print(data.shape)
data.head()

(1285984, 47)


,_individual,_decision,_choice,_selected,tag_key,h3_index,time,depth_bin,n_depth_bin,cos_moon,...,log_odds,odds,probability,experiment_name,run_id,_train,_partition,month,max_depth_bin,radians
0,73,318543,3,False,159014b,840c539ffffffff,2017-05-09,75.0,0.3,-0.894049,...,-2.148106,0.116705,0.066965,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,1.259794
1,73,318543,2,False,159014b,840c539ffffffff,2017-05-09,50.0,0.2,-0.894049,...,-1.385950,0.250086,0.143499,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,1.259794
2,73,318543,1,True,159014b,840c539ffffffff,2017-05-09,25.0,0.1,-0.894049,...,0.319166,1.375980,0.789536,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,1.259794
3,73,318571,3,False,159014b,840c539ffffffff,2017-05-11,75.0,0.3,-0.999360,...,-2.273380,0.102964,0.063244,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,2.953848
4,73,318571,1,True,159014b,840c539ffffffff,2017-05-11,25.0,0.1,-0.999360,...,0.243417,1.275600,0.783521,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,2.953848


In [3]:
key_info = {
    'radians': np.pi / 4,
    'month': 1,
    'max_depth_bin': 0.1,
}
key_cols = []
for key, bound in key_info.items():
    data[f'key_{key}'] = round(data[key] / bound).astype(int)
    key_cols.append(f'key_{key}')
data.head()

,_individual,_decision,_choice,_selected,tag_key,h3_index,time,depth_bin,n_depth_bin,cos_moon,...,experiment_name,run_id,_train,_partition,month,max_depth_bin,radians,key_radians,key_month,key_max_depth_bin
0,73,318543,3,False,159014b,840c539ffffffff,2017-05-09,75.0,0.3,-0.894049,...,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,1.259794,2,5,3
1,73,318543,2,False,159014b,840c539ffffffff,2017-05-09,50.0,0.2,-0.894049,...,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,1.259794,2,5,3
2,73,318543,1,True,159014b,840c539ffffffff,2017-05-09,25.0,0.1,-0.894049,...,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,1.259794,2,5,3
3,73,318571,3,False,159014b,840c539ffffffff,2017-05-11,75.0,0.3,-0.999360,...,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,2.953848,4,5,3
4,73,318571,1,True,159014b,840c539ffffffff,2017-05-11,25.0,0.1,-0.999360,...,chinook-depth-3-1-18,dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed5...,False,3,5,0.3,2.953848,4,5,3


In [4]:
def get_likelihoods(x):
    likelihoods = {}
    for _, row in x.iterrows():
        likelihoods[row['depth_bin']] = row['probability']
    return likelihoods

odf = data.groupby(['_individual', '_decision'] + key_cols).apply(get_likelihoods).reset_index().rename(columns={0: 'likelihoods'})
odf

/tmp/ipykernel_62508/1204984729.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  odf = data.groupby(['_individual', '_decision'] + key_cols).apply(get_likelihoods).reset_index().rename(columns={0: 'likelihoods'})


,_individual,_decision,key_radians,key_month,key_max_depth_bin,likelihoods
0,73,318534,-3,5,3,"{25.0: 0.7363764047622681, 50.0: 0.17356325685..."
1,73,318535,-3,5,3,"{50.0: 0.15043522417545319, 75.0: 0.0777639970..."
2,73,318536,2,5,3,"{75.0: 0.06450760364532471, 50.0: 0.1397746652..."
3,73,318537,-4,5,3,"{25.0: 0.7822081446647644, 50.0: 0.14714074134..."
4,73,318538,4,5,3,"{50.0: 0.15154165029525757, 25.0: 0.7844128608..."
...,...,...,...,...,...,...
192638,110,521549,-2,8,7,"{50.0: 0.14301204681396484, 150.0: 0.028962479..."
192639,110,521550,-2,8,7,"{25.0: 0.6855514645576477, 50.0: 0.13995723426..."
192640,110,521551,-2,8,7,"{250.0: 0.0010565487900748849, 25.0: 0.6695109..."
192641,110,521552,1,8,7,"{150.0: 0.1539866179227829, 250.0: 0.010730594..."


In [5]:
likelihoods = defaultdict(list)
for _, row in tqdm(odf.iterrows(), total=odf.shape[0]):
    key = tuple(row[key] for key in key_cols)
    likelihoods[key].append(row['likelihoods'])

100%|██████████| 192643/192643 [00:05<00:00, 35896.34it/s]


In [6]:
decisions = data[['_individual', '_decision'] + key_cols].drop_duplicates()
permutations = []
for i in range(1):
    print(f"Permutation {i}")
    rows = []
    for _, row in tqdm(decisions.iterrows(), total=decisions.shape[0]):
        key = tuple(row[key] for key in key_cols)
        likelihood = choice(likelihoods[key])
        for depth_bin, prob in likelihood.items():
            rows.append({
                '_individual': row['_individual'],
                '_decision': row['_decision'],
                'depth_bin': depth_bin,
                'probability': prob,
                **{key: row[key] for key in key_cols},
            })
    perm = pd.DataFrame(rows).merge(data[['_individual', '_decision', 'depth_bin', '_selected', 'salinity', 'mixed_layer_thickness', 'nitrate']], on=['_individual', '_decision', 'depth_bin'], how='inner')
    perm['permutation'] = i
    permutations.append(perm)
perm = pd.concat(permutations, ignore_index=True)
print(perm.shape)
perm.head()

Permutation 0


100%|██████████| 192643/192643 [00:15<00:00, 12613.88it/s]


(1285984, 12)


,_individual,_decision,depth_bin,probability,key_radians,key_month,key_max_depth_bin,_selected,salinity,mixed_layer_thickness,nitrate,permutation
0,73,318543,50.0,0.137981,2,5,3,False,31.977227,10.741591,5.509418,0
1,73,318543,75.0,0.068546,2,5,3,False,32.031129,10.730308,6.687875,0
2,73,318543,25.0,0.793473,2,5,3,True,31.926942,10.741591,2.355515,0
3,73,318571,25.0,0.754404,4,5,3,True,31.906853,10.528886,1.979406,0
4,73,318571,50.0,0.162939,4,5,3,False,31.987227,10.528886,5.348124,0


In [7]:
perm['nll'] = -np.log(perm['probability'])
data['nll'] = -np.log(data['probability'])
perm[perm['_selected']]['nll'].mean(), data[data['_selected']]['nll'].mean()

(1.430024857187774, 1.412636)

In [8]:
pdf = perm[perm['_selected']].groupby(key_cols)['nll'].mean().reset_index()
adf = data[data['_selected']].groupby(key_cols)['nll'].mean().reset_index()
df = pdf.merge(adf, on=key_cols, how='inner', suffixes=('_perm', '_data'))
df['nll_diff'] = df['nll_perm'] - df['nll_data']
df.sort_values('nll_diff', ascending=False, inplace=True)
df

,key_radians,key_month,key_max_depth_bin,nll_perm,nll_data,nll_diff
730,3,11,7,1.988439,1.735626,0.252813
637,2,11,7,2.022228,1.774383,0.247845
545,1,11,7,1.975333,1.754901,0.220432
832,4,12,7,1.805002,1.601512,0.203490
559,2,1,4,1.418617,1.220823,0.197794
...,...,...,...,...,...,...
723,3,10,9,1.745924,1.840515,-0.094591
630,2,10,9,1.785331,1.895517,-0.110186
655,3,1,9,1.954105,2.075285,-0.121180
446,0,10,9,1.858819,2.003936,-0.145117


In [10]:
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
sdf = pd.DataFrame({
    'quantile': quantiles,
    'nll_diff': [df['nll_diff'].quantile(q) for q in quantiles],
})
sdf

,quantile,nll_diff
0,0.10,-0.025379
1,0.25,-0.005248
2,0.50,0.007321
3,0.75,0.029937
4,0.90,0.066370


In [22]:
df[df['nll_diff'] > 0.05].shape[0] / df.shape[0]

0.15430622009569378

In [23]:
df[df['nll_diff'] < -0.05].shape[0] / df.shape[0]

0.03827751196172249

In [11]:
_filter = data.groupby(key_cols)['_individual'].nunique().reset_index().sort_values('_individual', ascending=True)
_filter = _filter[_filter['_individual'] > 6]
fdf = df.merge(_filter[key_cols])
fdf

,key_radians,key_month,key_max_depth_bin,nll_perm,nll_data,nll_diff
0,-2,10,6,1.788793,1.611243,0.177550
1,-1,10,6,1.737051,1.577474,0.159578
2,-3,10,6,1.729478,1.584806,0.144672
3,2,10,6,1.775370,1.666821,0.108549
4,0,10,6,1.749345,1.650246,0.099098
...,...,...,...,...,...,...
96,-1,12,10,1.882925,1.946420,-0.063495
97,-3,12,10,1.839070,1.903810,-0.064740
98,1,12,10,1.989644,2.055890,-0.066246
99,2,10,5,1.667813,1.736835,-0.069023


In [12]:
fdf[fdf['nll_diff'] < -0.05].shape[0] / df.shape[0]

0.009569377990430622

In [ ]:
data.merge(
    df[df['nll_diff'] < 0.00][['key_radians', 'key_month', 'key_max_depth_bin']], 
    how='inner'
).shape[0] / data.shape[0]

0.3162932042700376

: 

In [13]:
101/836

0.12081339712918661

In [65]:
df[(df['key_month'] == 1) & (df['key_max_depth_bin'].isin([4]))].groupby(['key_sin_sun'])['nll_diff'].describe()

,count,mean,std,min,25%,50%,75%,max
key_sin_sun,,,,,,,,
-2,5.0,-0.014303,0.031225,-0.047757,-0.031220,-0.029929,0.012665,0.024725
-1,2.0,-0.000065,0.011117,-0.007926,-0.003996,-0.000065,0.003866,0.007796
0,2.0,0.001670,0.038104,-0.025274,-0.011802,0.001670,0.015142,0.028614
1,2.0,0.058573,0.019780,0.044586,0.051579,0.058573,0.065566,0.072559
2,5.0,0.146262,0.050311,0.096516,0.108858,0.125897,0.194688,0.205349


In [19]:
perm[perm['_selected']].merge(_filter[key_cols])['nll'].mean(), data[data['_selected']].merge(_filter[key_cols])['nll'].mean()

(1.53286135135977, 1.5194967)

In [20]:
np.exp(-1.5194967)

0.21882199235061708

In [21]:
np.exp(-1.53286135135977)

0.2159169682696969

In [70]:
pdf = perm[(perm['key_max_depth_bin'] == 7) & (perm['key_month'] == 11) & (perm['key_sin_sun'] >= 0)]
adf = data[(data['key_max_depth_bin'] == 7) & (data['key_month'] == 11) & (data['key_sin_sun'] >= 0)]
df = pdf.merge(adf[['_individual', '_decision', 'depth_bin', 'nll']], on=['_individual', '_decision', 'depth_bin'], how='inner', suffixes=('_perm', '_data'))
df['nll_diff'] = df['nll_perm'] - df['nll_data']
df['max_salinity'] = df.groupby(['_individual', '_decision'])['salinity'].transform('max')
df['min_salinity'] = df.groupby(['_individual', '_decision'])['salinity'].transform('min')
df['salinity_diff'] = df['max_salinity'] - df['min_salinity']
df['binned_salinity_diff'] = round(df['salinity_diff'] / 0.5) * 0.5
df['binned_salinity'] = round(df['salinity'] / 0.5) * 0.5
px.scatter(
    df[df['_selected']].groupby(['binned_salinity'])['nll_diff'].mean().reset_index(),
    x='binned_salinity', y='nll_diff',
)

In [83]:
adf = data[(data['key_max_depth_bin'] == 7) & (data['key_month'] == 11) & (data['key_sin_sun'] >= 0)]
adf['max_salinity'] = adf.groupby(['_individual', '_decision'])['salinity'].transform('max')
adf['min_salinity'] = adf.groupby(['_individual', '_decision'])['salinity'].transform('min')
adf['stratified'] = (adf['max_salinity'] - adf['min_salinity']) >= 1
adf[adf['_selected'] & (adf['stratified'])].groupby(['depth_bin']).size()

/tmp/ipykernel_14924/834438279.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/834438279.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/834438279.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



depth_bin
25.0     411
50.0     226
75.0      48
100.0     56
150.0    195
200.0    176
dtype: int64

In [84]:
adf[adf['_selected'] & (~adf['stratified'])].groupby(['depth_bin']).size()

depth_bin
25.0      68
50.0      27
75.0     305
100.0    353
150.0     81
dtype: int64

In [191]:
adf = data[(data['key_max_depth_bin'] > 4) & (data['key_month'] == 8) & (data['key_sin_sun'] >= 0)]
adf['max_salinity'] = adf.groupby(['_individual', '_decision'])['salinity'].transform('max')
adf['min_salinity'] = adf.groupby(['_individual', '_decision'])['salinity'].transform('min')
adf['saline'] = adf['min_salinity'] >= 32

saline = pd.DataFrame((adf[adf['_selected'] & (adf['saline'])].groupby(['depth_bin']).size() / adf[adf['_selected'] & (adf['saline'])].shape[0])).rename(columns={0: 'proportion'}).reset_index()
saline['proportion'] = round(saline['proportion'], 2)
fresh = pd.DataFrame((adf[adf['_selected'] & (~adf['saline'])].groupby(['depth_bin']).size() / adf[adf['_selected'] & (~adf['saline'])].shape[0])).rename(columns={0: 'proportion'}).reset_index()
fresh['proportion'] = round(fresh['proportion'], 2)
saline.merge(fresh, on='depth_bin', how='inner', suffixes=('_saline', '_fresh'))

/tmp/ipykernel_14924/567468083.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/567468083.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/567468083.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,depth_bin,proportion_saline,proportion_fresh
0,25.0,0.21,0.25
1,50.0,0.17,0.31
2,75.0,0.14,0.28
3,100.0,0.16,0.13
4,150.0,0.24,0.03
5,200.0,0.08,0.00


In [194]:
perm[perm['_selected'] & (perm['key_sin_sun'] <= 0) & (perm['salinity'] <= 32)]['nll'].mean(), data[data['_selected'] & (data['key_sin_sun'] <= 0) & (data['salinity'] <= 32)]['nll'].mean()

(0.9999435535538539, 0.98904234)

In [199]:
(
    perm[perm['_selected'] & (perm['key_sin_sun'] >= 0) & (perm['salinity'] > 32) & (perm['key_max_depth_bin'] > 6) & (perm['key_month'] == 11)]['nll'].mean(), 
    data[data['_selected'] & (data['key_sin_sun'] >= 0) & (data['salinity'] > 32) & (data['key_max_depth_bin'] > 6) & (perm['key_month'] == 11)]['nll'].mean()
)

(1.6904150061977041, 1.6123723)

In [210]:
df = data[(data['depth_bin'] == 25.0) & (data['key_max_depth_bin'].isin([6, 7, 8])) & (data['key_sin_sun'] >= 0)]
df['min_salinity'] = df.groupby(['_individual', '_decision'])['salinity'].transform('min')
df['binned_min_salinity'] = round(df['min_salinity'] / 0.25) * 0.25
px.scatter(
    df.groupby(['month', 'binned_min_salinity'])['_selected'].agg(['mean', 'count']).reset_index(),
    x='binned_min_salinity', y='mean',  size='count', facet_col='month', category_orders={'month': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]},
    facet_col_wrap=4
)

/tmp/ipykernel_14924/1702199058.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/1702199058.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [232]:
data['max_salinity'] = data.groupby(['_individual', '_decision'])['salinity'].transform('max')
data['min_salinity'] = data.groupby(['_individual', '_decision'])['salinity'].transform('min')
df = data[(data['depth_bin'] == 25.0) & (data['key_max_depth_bin'].isin([6, 7, 7]))]
df['radians'] = np.arctan2(df['sin_sun'], df['cos_sun'])
df['radians'] = round(df['radians'] * 5) / 5
df['salinity_diff'] = df['max_salinity'] - df['min_salinity']
df['stratified'] = df['salinity_diff'] >= 1
df = df.groupby(['radians', 'stratified', 'month'])['_selected'].agg(['mean', 'count']).reset_index()
px.scatter(
    df, x='radians', y='mean', color='stratified',
    category_orders={'stratified': [True, False], 'month': list(range(1, 13))},
    color_discrete_sequence=['aquamarine', 'navy'],
    facet_col='month', facet_col_wrap=4,
    height=900, width=1000
)

/tmp/ipykernel_14924/1036238160.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/1036238160.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_14924/1036238160.py:6: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_1

In [226]:
df = data[data['depth_bin'] == 25.0]
df.columns

Index(['_individual', '_decision', '_choice', '_selected', 'tag_key',
       'h3_index', 'time', 'depth_bin', 'n_depth_bin', 'cos_moon', 'sin_moon',
       'cos_orbit', 'sin_orbit', 'cos_sun', 'sin_sun', 'chlorophyll',
       'net_primary_production', 'nitrate', 'oxygen', 'phosphate', 'silicate',
       'n_chlorophyll', 'n_net_primary_production', 'n_nitrate', 'n_oxygen',
       'n_phosphate', 'n_silicate', 'elevation', 'mixed_layer_thickness',
       'salinity', 'temperature', 'n_elevation', 'n_mixed_layer_thickness',
       'n_salinity', 'n_temperature', 'velocity_east', 'velocity_north',
       'log_odds', 'odds', 'probability', 'experiment_name', 'run_id',
       '_train', '_partition', 'month', 'max_depth_bin', 'key_cos_sun',
       'key_sin_sun', 'key_month', 'key_max_depth_bin', 'nll'],
      dtype='object')

In [177]:
adf[adf['_selected'] & (~adf['saline'])].groupby(['depth_bin']).size() / adf[adf['_selected'] & (~adf['saline'])].shape[0]

depth_bin
25.0     0.550898
50.0     0.251497
75.0     0.053892
100.0    0.023952
150.0    0.059880
200.0    0.059880
dtype: float64

In [148]:
data[data['depth_bin'] == 100]['salinity'].describe()

count    175329.000000
mean         32.651082
std           0.363638
min          31.331065
25%          32.426962
50%          32.698403
75%          32.905850
max          33.694808
Name: salinity, dtype: float64

In [53]:
space = data[['h3_index', 'key_max_depth_bin']].drop_duplicates()
space['lat'] = space['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
space['lon'] = space['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])
space['size'] = 0.01
fig = px.scatter_mapbox(
    space[space['key_max_depth_bin'].isin([6, 7])],
    lat='lat',
    lon='lon',
    color='key_max_depth_bin',  # Color points by probability
    size='size',  # Adjust as needed
    size_max=11,  # Adjust as needed
    zoom=4,  # Adjust zoom level
    mapbox_style="carto-positron",  # Choose a map style,
    #range_color=[0.0, 0.3],
    height=600,
    width=1000
)
fig.show()